# A/B test an onboarding prompt and ship the winner

Your Ridgeline Outfitters onboarding assistant knows the policy but replies in full sentences. Its caller needs the exact policy value to fill a form. Commit a baseline, use Agent Learning Kit to propose a shorter prompt, and compare both **stored versions** on employee questions the optimizer did not see. Move a demo serving label only if the candidate passes the release rule, then verify a rollback.

[Open in Colab](https://colab.research.google.com/github/future-agi/cookbooks/blob/main/use-cases/full-prompt-lifecycle.ipynb) · [Prompt versioning guide](https://docs.futureagi.com/docs/cookbook/quickstart/prompt-versioning)

Use a fresh Python 3.12 runtime, a Future AGI workspace where you can create prompts, and an OpenAI key. The optimizer and comparison make billable model calls. All policies and questions below are fictional.

## Install the two SDKs separately

The notebook kernel runs Agent Learning Kit for optimization and scoring. A separate virtual environment runs the Future AGI platform SDK for prompt management. Both distributions install modules under `fi`, so do not install them together.

In [ ]:
%pip install agent-learning-kit==0.3.7 virtualenv

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path
from uuid import uuid4

work = Path(tempfile.mkdtemp(prefix="onboarding-prompts-"))
os.chdir(work)
subprocess.run([sys.executable, "-m", "virtualenv", "--quiet", str(work / "platform")], check=True)
platform_python = work / "platform" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
subprocess.run([str(platform_python), "-m", "pip", "install", "--quiet", "futureagi==0.8.0"], check=True)
os.environ["ONBOARDING_PROMPT"] = f"ridgeline-onboarding-{uuid4().hex[:12]}"
print("Demo prompt:", os.environ["ONBOARDING_PROMPT"])

In [ ]:
import os
from getpass import getpass

def load_secret(name):
    if os.environ.get(name):
        return
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except (ImportError, KeyError, RuntimeError):
        value = None
    except Exception as exc:
        if type(exc).__name__ not in {"SecretNotFoundError", "NotebookAccessError"}:
            raise
        value = None
    value = value or getpass(f"{name}: ")
    if not value:
        raise ValueError(f"{name} is required")
    os.environ[name] = value

for name in ['FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY']:
    load_secret(name)

## Store v1 and attach a demo serving label

The helper below runs **only** in the platform environment. Each invocation starts a new process so a previous label lookup cannot mask a move. `fetch` reads the text from the saved message parts, not from a local copy. `onboarding-demo-live` is a custom label on the new prompt, not your production deployment.

In [ ]:
%%writefile store_prompt.py
import json
import os
import sys
from pathlib import Path
from fi.prompt import Prompt
from fi.prompt.types import ModelConfig, PromptTemplate, SystemMessage, UserMessage

name = os.environ["ONBOARDING_PROMPT"]
label = "onboarding-demo-live"
action, *args = sys.argv[1:]

def template():
    text = Path("instruction.txt").read_text().replace("{case}", "{{employee_case}}")
    return PromptTemplate(name=name, messages=[
        SystemMessage(content="You help new employees by answering from the supplied onboarding policy."),
        UserMessage(content=text)],
                          model_configuration=ModelConfig(model_name="gpt-4o-mini", temperature=0))

if action == "baseline":
    prompt = Prompt(template=template())
    prompt.create()
    if not prompt.get_label_by_name(label):
        prompt.create_label(label)
    prompt.commit_current_version(message="Baseline onboarding answers", label=label)
elif action == "candidate":
    prompt = Prompt.get_template_by_name(name=name, version="v1")
    prompt.create_new_version(template=template())
    prompt.commit_current_version(message="Candidate for exact policy values")
elif action == "label":
    Prompt.assign_label_to_template_version(template_name=name, version=args[0], label=label)
elif action == "fetch":
    selector = {"label": label} if args[0] == "live" else {"version": args[0]}
    prompt = Prompt.get_template_by_name(name=name, **selector)
    messages = []
    for message in prompt.template.model_dump()["messages"]:
        content = message["content"]
        if isinstance(content, list):
            if any(part["type"] != "text" for part in content):
                raise ValueError("This example only renders text")
            content = "".join(part["text"] for part in content)
        messages.append({"role": message["role"], "content": content})
    Path("snapshot.json").write_text(json.dumps({"version": prompt.template.version, "messages": messages}))
else:
    raise ValueError(f"Unknown action: {action}")

In [ ]:
import json

def store(action, *args, instruction=None):
    if instruction is not None:
        Path("instruction.txt").write_text(instruction)
    subprocess.run([str(platform_python), "store_prompt.py", action, *args], check=True)
    if action == "fetch":
        return json.loads(Path("snapshot.json").read_text())

seed = "Answer the employee's question in a friendly full sentence using the given policy.\n{case}"
store("baseline", instruction=seed)
baseline = store("fetch", "live")
assert baseline["version"] == "v1" and "{{employee_case}}" in baseline["messages"][-1]["content"]
print("Serving", baseline["version"])

## Measure v1 on the training questions

Each policy is a short record of named fields. Each case contains that record and an employee question. The form accepts only the value of the field the question refers to, without its name or any added explanation. The expected field value is only used for scoring. A full sentence can be factually correct and still fail this output contract.

In [ ]:
from openai import OpenAI
from fi.evals import evaluate

client = OpenAI()
train = [
    {"case": "Policy: Enrollment window = 30 days; Plan change fee = $0. Question: How long do I have to enroll?", "answer": "30 days"},
    {"case": "Policy: Badge desk opens = 8 AM; Badge desk closes = 5 PM. Question: What time does the desk open?", "answer": "8 AM"},
]

def compare(snapshot, examples):
    scores = []
    for row in examples:
        messages = [{"role": m["role"], "content": m["content"].replace("{{employee_case}}", row["case"])}
                    for m in snapshot["messages"]]
        reply = client.chat.completions.create(model="gpt-4o-mini", temperature=0,
                                               messages=messages)
        answer = reply.choices[0].message.content
        if not isinstance(answer, str) or not answer.strip():
            raise ValueError("Expected a text reply")
        grade = evaluate("equals", engine="local", output=answer, expected_response=row["answer"],
                         config={"case_sensitive": True})
        if grade.error:
            raise RuntimeError(grade.error)
        scores.append(bool(grade.passed))
        print(snapshot["version"], "PASS" if grade.passed else "FAIL", repr(answer), "expected", repr(row["answer"]))
    return sum(scores)

print("v1 training score:", compare(baseline, train), "/", len(train))

## Optimize a candidate, then commit v2 without moving the label

The teacher sees the two training rows. Two rounds let the optimizer score its first rewrite. Inspect its selected template: it must still have a single `{case}` field. If it keeps the seed or breaks that contract, use the explicit candidate below and identify it as a manual alternative, not an optimizer win.

In [ ]:
from string import Formatter
from fi.evals.metrics import Equals
from fi.opt.base import Evaluator
from fi.opt.datamappers import BasicDataMapper
from fi.opt.generators import LiteLLMGenerator
from fi.opt.optimizers import MetaPromptOptimizer

model = "openai/gpt-4o-mini"
optimizer = MetaPromptOptimizer(teacher_generator=LiteLLMGenerator(model, "{prompt}"), task_model=model)
result = optimizer.optimize(
    evaluator=Evaluator(metric=Equals(config={"case_sensitive": True})),
    data_mapper=BasicDataMapper({"response": "generated_output", "expected_response": "answer"}),
    dataset=train, initial_prompts=[seed], num_rounds=2, eval_subset_size=len(train),
    task_description="Select the policy field requested by the employee. Copy just its value after the equals sign, without the field name, surrounding punctuation, or explanation. Keep {case} as the only placeholder.",
)
if not result.history:
    raise RuntimeError("No prompt was scored. Check your model connection before continuing.")
for number, entry in enumerate(result.history, start=1):
    print(f"Round {number} training score: {entry.average_score:.0%}")
candidate = result.best_generator.get_prompt_template()
try:
    fields = [(field, spec, conv) for _, field, spec, conv in Formatter().parse(candidate) if field is not None]
except ValueError:
    fields = []
if candidate == seed or fields != [("case", "", None)] or candidate.count("{case}") != 1:
    candidate = "Select the field that answers the question. Return ONLY its exact value after the equals sign. Do not include the field name, other fields, separators, punctuation at the end of the record, or any extra words.\n{case}"
    print("Using the manual exact-value alternative.")
else:
    print("Using the optimizer's selected instruction.")
print(candidate)

In [ ]:
store("candidate", instruction=candidate)
v1 = store("fetch", "v1")
v2 = store("fetch", "v2")
assert v1 == baseline and v2["version"] == "v2"
assert v2["messages"][-1]["content"] == candidate.replace("{case}", "{{employee_case}}")
assert store("fetch", "live") == v1
print("Committed", v2["version"], "while serving", v1["version"])

## Compare the saved versions on held-out questions

Ask both versions the same new questions with the same model and temperature. This is a small offline release check, not a live traffic experiment or a statistical guarantee. Promote v2 only if it beats v1 and answers both questions exactly.

In [ ]:
holdout = [
    {"case": "Policy: Remote laptop shipping lead time = 5 business days; Badge delivery lead time = 2 days. Question: How far in advance do remote laptops ship?", "answer": "5 business days"},
    {"case": "Policy: Dental plan = Premium; Medical-only plan = Basic. Question: Which plan includes dental?", "answer": "Premium"},
]
scores = {row["version"]: compare(row, holdout) for row in (v1, v2)}
print("Held-out exact matches:", scores)
winner = "v2" if scores["v2"] == len(holdout) and scores["v2"] > scores["v1"] else "v1"

## Serve the winner and verify rollback

Apply the gate to the demo label. Fetch it again by name: this is the version an application requesting `onboarding-demo-live` would serve. Restore the baseline at the end. Both numbered versions remain available for inspection in Prompt Management.

In [ ]:
if winner == "v2":
    store("label", "v2")
    print("Candidate passed; moved the demo serving label.")
else:
    print("Candidate did not clear the gate; kept the baseline.")
served = store("fetch", "live")
assert served == {"v1": v1, "v2": v2}[winner]
print("Serving", served["version"])
if winner == "v2":
    store("label", "v1")
assert store("fetch", "live") == v1
assert store("fetch", "v2") == v2
print("Final label: v1. Candidate history: v2.")